# Make or Miss League?

In this project we will try to compare the predictive power of scoring statistics and non-scoring statistics on the outcome of an NBA basketball game.


We will create 3 models which we will use to predict the last 72 games of an NBA teams season, based on their first 10 games:

- Full-scope model (includes all statistics, including scoring)

- Scoring model (Only includes statistics about the team's offense)

- Non-scoring model (include all statistics except those relating to the team's offense)

Will non-scoring statistics be able to predict the success of an NBA team, or does winning in this league really come down to makes and misses?

SAMPLING NOTES:

What statistics we should use to predict any given game is kind of subjective. Using the season's cumulative statistics helps negate variance but could be unrepresentative of the actual current team if injuries or trades occurred.

To avoid this misrepresentation of teams, we can create a rolling average which uses the past 10 games statistics to predict the next game. This does introduce noise but may be more effective.

We can test which method works best.

SCORING MODEL NOTES:

Ideally, the model only reflects the scoring/offensive effectiveness of the team itself, and doesn't take in any variables which indicate defense or opponent's offensive success.

Essentially testing the predictive power of offensive effectiveness, independent of defense.

NON-SCORING MODEL NOTES:

Ideally, the model depicts everything about basketball except whether or not the team in question can put the ball in the basket. Completely exclude all variables indicating points or whether a basket was made (leads, ties, assists (assistPercentage can probably stay), shooting percentages, attempts etc.)

Opponent scoring stuff can probably stay

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
# Import NBA team statistics since Nov. 1st 1996

import kagglehub

# Download latest version
path = kagglehub.dataset_download("eoinamoore/historical-nba-data-and-player-box-scores")

print("Path to dataset files:", path)

100%|██████████| 1.03G/1.03G [01:37<00:00, 11.4MB/s]

Extracting files...


Path to dataset files: C:\Users\ragha\.cache\kagglehub\datasets\eoinamoore\historical-nba-data-and-player-box-scores\versions\496


In [3]:
nba_team_stats_df = pd.read_csv(path + "/TeamStatisticsExtended.csv")

In [4]:
nba_team_stats_df = nba_team_stats_df[nba_team_stats_df['gameType'] == 'Regular Season']
nba_team_stats_df.head(5)

,gameId,gameDateTimeEst,gameType,gameLabel,gameSubLabel,seriesGameNumber,teamId,teamCity,teamName,opponentTeamId,...,percentUnassisted2PointMade,percentAssisted3PointMade,percentUnassisted3PointMade,percentAssistedFieldGoalsMade,percentUnassistedFieldGoalsMade,freeThrowAttemptRate,opponentEffectiveFieldGoalPercentage,opponentFreeThrowAttemptRate,opponentTurnoverPercentage,opponentOffensiveReboundPercentage
168,22501193,2026-04-12 20:30:00,Regular Season,NaN,NaN,NaN,1610612741,Chicago,Bulls,1610612742,...,0.524,1.000,0.000,0.577,0.423,0.179,0.630,0.270,0.162,0.365
169,22501193,2026-04-12 20:30:00,Regular Season,NaN,NaN,NaN,1610612742,Dallas,Mavericks,1610612741,...,0.533,0.955,0.045,0.673,0.327,0.270,0.538,0.179,0.116,0.288
170,22501194,2026-04-12 20:30:00,Regular Season,NaN,NaN,NaN,1610612763,Memphis,Grizzlies,1610612745,...,0.423,0.615,0.385,0.590,0.410,0.105,0.524,0.286,0.107,0.452
171,22501194,2026-04-12 20:30:00,Regular Season,NaN,NaN,NaN,1610612745,Houston,Rockets,1610612763,...,0.647,0.786,0.214,0.479,0.521,0.286,0.479,0.105,0.126,0.179
172,22501195,2026-04-12 20:30:00,Regular Season,NaN,NaN,NaN,1610612740,New Orleans,Pelicans,1610612750,...,0.625,1.000,0.000,0.444,0.556,0.355,0.562,0.483,0.102,0.308


In [5]:
nba_team_stats_df = nba_team_stats_df.drop(columns=['gameType', 'gameLabel', 'gameSubLabel', 'seriesGameNumber'])
nba_team_stats_df

,gameId,gameDateTimeEst,teamId,teamCity,teamName,opponentTeamId,opponentTeamCity,opponentTeamName,home,win,...,percentUnassisted2PointMade,percentAssisted3PointMade,percentUnassisted3PointMade,percentAssistedFieldGoalsMade,percentUnassistedFieldGoalsMade,freeThrowAttemptRate,opponentEffectiveFieldGoalPercentage,opponentFreeThrowAttemptRate,opponentTurnoverPercentage,opponentOffensiveReboundPercentage
168,22501193,2026-04-12 20:30:00,1610612741,Chicago,Bulls,1610612742,Dallas,Mavericks,0,0,...,0.524,1.000,0.000,0.577,0.423,0.179,0.630,0.270,0.162,0.365
169,22501193,2026-04-12 20:30:00,1610612742,Dallas,Mavericks,1610612741,Chicago,Bulls,1,1,...,0.533,0.955,0.045,0.673,0.327,0.270,0.538,0.179,0.116,0.288
170,22501194,2026-04-12 20:30:00,1610612763,Memphis,Grizzlies,1610612745,Houston,Rockets,0,0,...,0.423,0.615,0.385,0.590,0.410,0.105,0.524,0.286,0.107,0.452
171,22501194,2026-04-12 20:30:00,1610612745,Houston,Rockets,1610612763,Memphis,Grizzlies,1,1,...,0.647,0.786,0.214,0.479,0.521,0.286,0.479,0.105,0.126,0.179
172,22501195,2026-04-12 20:30:00,1610612740,New Orleans,Pelicans,1610612750,Minnesota,Timberwolves,0,0,...,0.625,1.000,0.000,0.444,0.556,0.355,0.562,0.483,0.102,0.308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79705,29600005,1996-11-01 19:30:00,1610612737,Atlanta,Hawks,1610612748,Miami,Heat,0,0,...,0.571,1.000,0.000,0.520,0.480,0.492,0.513,0.244,0.207,0.326
79706,29600007,1996-11-01 19:30:00,1610612765,Detroit,Pistons,1610612754,Indiana,Pacers,1,1,...,0.769,1.000,0.000,0.375,0.625,0.515,0.486,0.311,0.196,0.311
79707,29600007,1996-11-01 19:30:00,1610612754,Indiana,Pacers,1610612765,Detroit,Pistons,0,0,...,0.481,0.833,0.167,0.576,0.424,0.311,0.515,0.515,0.209,0.333
79708,29600001,1996-11-01 19:00:00,1610612738,Boston,Celtics,1610612741,Chicago,Bulls,1,0,...,0.533,0.750,0.250,0.526,0.474,0.274,0.574,0.432,0.198,0.306


In [6]:
nba_team_stats_df.columns

Index(['gameId', 'gameDateTimeEst', 'teamId', 'teamCity', 'teamName',
       'opponentTeamId', 'opponentTeamCity', 'opponentTeamName', 'home', 'win',
       'teamScore', 'opponentScore', 'seed', 'numMinutes', 'assists', 'steals',
       'blocks', 'blocksAgainst', 'fieldGoalsMade', 'fieldGoalsAttempted',
       'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted',
       'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted',
       'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive',
       'reboundsTotal', 'reboundsTeam', 'foulsPersonal', 'personalFoulsDrawn',
       'turnovers', 'turnoversTeam', 'plusMinusPoints', 'q1Points', 'q2Points',
       'q3Points', 'q4Points', 'ot1Points', 'ot2Points', 'otAllPoints',
       'benchPoints', 'biggestLead', 'biggestScoringRun', 'leadChanges',
       'pointsFastBreak', 'pointsFromTurnovers', 'pointsInThePaint',
       'pointsSecondChance', 'timesTied', 'timeoutsRemaining', 'seasonWins',
       'seasonLosse

FULL SCOPE MODEL NOTES:

We want pretty much every variable available, but I think we should eliminate variables which are proxies for winning or losing

NON-SCORING MODEL NOTES:

Ideally, the model depicts everything about basketball except whether or not the team in question can put the ball in the basket. Completely exclude all variables indicating points or whether a basket was made (leads, ties, assists (assistPercentage can probably stay), shooting percentages, attempts etc.)

Opponent scoring stuff can probably stay

SCORING MODEL NOTES:

Ideally, the model only reflects the scoring/offensive effectiveness of the team itself, and doesn't take in any variables which indicate defense or opponent's offensive success.

Essentially testing the predictive power of offensive effectiveness, independent of defense.

In [ ]:
# Metadata / identifiers which must be excluded from all models
metadata_cols = [
    'gameId', 'gameDateTimeEst', 'teamId', 'teamCity', 'teamName',
    'opponentTeamId', 'opponentTeamCity', 'opponentTeamName', 'seed',
    'numMinutes', 'timeoutsRemaining', 'seasonWins', 'seasonLosses'
]

# Target variable
target = 'win'

# Model 1: Full Scope
full_scope_cols = [
    # home court advantage
    'home',
    # Scoring
    'teamScore', 'opponentScore',
    'q1Points', 'q2Points', 'q3Points', 'q4Points',
    'ot1Points', 'ot2Points', 'otAllPoints',
    'benchPoints', 'plusMinusPoints',
    'pointsFastBreak', 'pointsFromTurnovers', 'pointsInThePaint',
    'pointsSecondChance', 'pointsOffTurnovers',
    'opponentPointsOffTurnovers', 'opponentPointsSecondChance',
    'opponentPointsFastBreak', 'opponentPointsInPaint',
    # Shooting
    'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
    'threePointersMade', 'threePointersAttempted', 'threePointersPercentage',
    'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage',
    'effectiveFieldGoalPercentage', 'trueShootingPercentage',
    'freeThrowAttemptRate', 'opponentEffectiveFieldGoalPercentage',
    'opponentFreeThrowAttemptRate',
    'percentFieldGoalAttempts2Point', 'percentFieldGoalAttempts3Point',
    'percentPoints2Point', 'percentPoints2PointMidRange',
    'percentPoints3Point', 'percentPointsFastBreak',
    'percentPointsFreeThrow', 'percentPointsOffTurnovers',
    'percentPointsInPaint',
    'percentAssisted2PointMade', 'percentUnassisted2PointMade',
    'percentAssisted3PointMade', 'percentUnassisted3PointMade',
    'percentAssistedFieldGoalsMade', 'percentUnassistedFieldGoalsMade',
    # Non-scoring
    'assists', 'steals', 'blocks', 'blocksAgainst',
    'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'reboundsTeam',
    'foulsPersonal', 'personalFoulsDrawn',
    'turnovers', 'turnoversTeam',
    'biggestLead', 'biggestScoringRun', 'leadChanges', 'timesTied',
    # Advanced
    'estimatedOffensiveRating', 'offensiveRating',
    'estimatedDefensiveRating', 'defensiveRating',
    'estimatedNetRating', 'netRating',
    'assistPercentage', 'assistToTurnoverRatio', 'assistRatio',
    'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage',
    'teamTurnoverPercentage', 'opponentTurnoverPercentage',
    'opponentOffensiveReboundPercentage',
    'estimatedPace', 'pace', 'pacePer40', 'possessions',
    'playerImpactEstimate'
]

# ── Model 2: Scoring Only ──────────────────────────────────────────────────────
scoring_cols = [
    'home',
    # Raw scoring
    'teamScore', #'opponentScore',
    'q1Points', 'q2Points', 'q3Points', 'q4Points',
    'ot1Points', 'ot2Points', 'otAllPoints',
    'benchPoints', #'plusMinusPoints',
    'pointsFastBreak', 'pointsFromTurnovers', 'pointsInThePaint',
    'pointsSecondChance', 'pointsOffTurnovers',
    #'opponentPointsOffTurnovers', 'opponentPointsSecondChance',
    #'opponentPointsFastBreak', 'opponentPointsInPaint',
    # Shooting
    'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
    'threePointersMade', 'threePointersAttempted', 'threePointersPercentage',
    'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage',
    'effectiveFieldGoalPercentage', 'trueShootingPercentage',
    'freeThrowAttemptRate', #'opponentEffectiveFieldGoalPercentage',
    #'opponentFreeThrowAttemptRate',
    'percentFieldGoalAttempts2Point', 'percentFieldGoalAttempts3Point',
    'percentPoints2Point', 'percentPoints2PointMidRange',
    'percentPoints3Point', 'percentPointsFastBreak',
    'percentPointsFreeThrow', 'percentPointsOffTurnovers',
    'percentPointsInPaint',
    'percentAssisted2PointMade', 'percentUnassisted2PointMade',
    'percentAssisted3PointMade', 'percentUnassisted3PointMade',
    'percentAssistedFieldGoalsMade', 'percentUnassistedFieldGoalsMade',
]



#THIS NEEDS MORE VARIABLES REGARDING OPPONENT SHOOTING STUFF
# ── Model 3: Non-Scoring Only
non_scoring_cols = [
    'home',
    # Hustle / defense
    'steals', 'blocks', 'blocksAgainst',
    'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'reboundsTeam',
    'foulsPersonal', 'personalFoulsDrawn',
    'turnovers', 'turnoversTeam',
    # Advanced non-scoring
    'assistPercentage', 'assistToTurnoverRatio', 'assistRatio',
    'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage',
    'teamTurnoverPercentage', 'opponentTurnoverPercentage',
    'opponentOffensiveReboundPercentage',
    'estimatedPace', 'pace', 'pacePer40', 'possessions',
]

**GENERAL NARRATIVE**

Current style of play is : offense is king -> 
* League is shifting towards optimizing pure offensive potential , Three point shooting, and high intensity offensive play
* 3 Models test independently for -> 
    * Full Scope model - Uses all columns to try and predicts win rate based off both offensive and non offensive metrics
    * Offensive (Scoring) Model -> Intends to isolate offense and win predictions (win rate)
    * Non Offensive -> Opposite columns from above offensive model to predict win rate

Our goal -> holistically analyze the data and model outputs across three in depth models, and compare results

Ask the question -> Is the NBA's craze over offensive efficacy truly warranted? (Is it the most optimizing metric)

**NOVELTY & NEW DIRECTION: The "Style Matchup" Predictor**

Instead of looking at teams in a vacuum, we want to frame the task around **stylistic clashes**. 
What happens when an "Elite Offense" plays an "Elite Defense"? 

**Approach:**
1. **Unsupervised Learning (Custering):** Group teams into playstyles first (e.g., Pure Offense, Gritty Defense, Balanced, Pace-Pushers).
2. **Exploratory Data Analysis (EDA):** Build a narrative identifying which historical and modern teams fit these archetypes. Are there interesting trends over time?
3. **Predictive Modeling:** Build a model to evaluate outcome probabilities when different styles collide. Which style historically prevails in a clash?

This reframes the problem from "predicting a win" to "analyzing the effectiveness of team building philosophies in head-to-head scenarios."